# MedScope 03: supervised fine-tuning

Trains two LoRA adapters on `Qwen/Qwen3.5-4B` from synthetic trajectories (no teacher model):
- **grounding**: answer + evidence box on answerable views, single turn
- **agentic**: all view kinds, multi-turn with zoom, refer and clarify

Then checks each on 40 validation cases. **Runtime:** L4 or A100, a few hours. Checkpoints go to Drive
every 50 steps; if Colab disconnects, run all cells again and training resumes.

**Before you start:** in the Colab sidebar, open the key icon (Secrets) and add:
- `HF_TOKEN`: a Hugging Face read token. For MedGemma, first accept its terms at https://huggingface.co/google/medgemma-1.5-4b-it
- `GITHUB_TOKEN`: only needed if the repo is private.

Turn on notebook access for both secrets when Colab asks.

In [ ]:
# --- setup: same in every notebook ---
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, json
DRIVE = '/content/drive/MyDrive/medscope'
WORK = '/content/work'
RESULTS = f'{DRIVE}/results'
ADAPTERS = f'{DRIVE}/adapters'
VIEWS = f'{WORK}/views'
for d in (DRIVE, RESULTS, ADAPTERS, WORK):
    os.makedirs(d, exist_ok=True)

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

gh = secret('GITHUB_TOKEN')
url = f'https://{gh}@github.com/bhaskargurram-ai/medscope' if gh else 'https://github.com/bhaskargurram-ai/medscope'
if os.path.exists('/content/medscope/.git'):
    !git -C /content/medscope pull -q
else:
    !git clone -q {url} /content/medscope
%cd /content/medscope
!git log --oneline -1
!pip install -q -e .

In [ ]:
# --- model libraries and Hugging Face login ---
!pip install -q -U transformers accelerate peft bitsandbytes
hf = secret('HF_TOKEN')
if hf:
    os.environ['HF_TOKEN'] = hf
    from huggingface_hub import login
    login(hf)
else:
    print('No HF_TOKEN secret: MedGemma (gated) will fail to download; Qwen works without it.')
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: Runtime > Change runtime type > L4 or A100')

In [ ]:
# --- restore the views built in notebook 01 (paths inside the manifests point to /content/work/views) ---
if not os.path.exists(f'{VIEWS}/test/views_clean.jsonl'):
    !tar -C {WORK} -xf {DRIVE}/slake/views.tar
!wc -l {VIEWS}/*/views_clean.jsonl

## Grounding adapter

In [ ]:
!python scripts/train_sft.py --model Qwen/Qwen3.5-4B --views {VIEWS}/train/views_clean.jsonl --mode grounding --out {ADAPTERS}/sft_grounding

## Agentic adapter

In [ ]:
!python scripts/train_sft.py --model Qwen/Qwen3.5-4B --views {VIEWS}/train/views_clean.jsonl --mode agentic --out {ADAPTERS}/sft_agentic

## Quick validation check

In [ ]:
for name in ['grounding', 'agentic']:
    !python scripts/evaluate.py --model Qwen/Qwen3.5-4B --adapter {ADAPTERS}/sft_{name}/final --views {VIEWS}/validation/views_clean.jsonl --out {RESULTS}/sft_{name}_val --name sft-{name} --split validation --limit-cases 40

In [ ]:
from medscope.colab import build_report, publish_doc, read_text, train_log_summary
from medscope.eval.runner import compare_runs
report = build_report('MedScope report - 03 SFT', [

    ('Grounding training loss', train_log_summary(f'{ADAPTERS}/sft_grounding/train_log.jsonl')),
    ('Agentic training loss', train_log_summary(f'{ADAPTERS}/sft_agentic/train_log.jsonl')),
    ('Leaderboard (all runs so far)', compare_runs(RESULTS)),
    ('Grounding adapter on validation', read_text(f'{RESULTS}/sft_grounding_val/report.md')),
    ('Agentic adapter on validation', read_text(f'{RESULTS}/sft_agentic_val/report.md')),

])
open(f'{RESULTS}/03_sft_report.md', 'w').write(report)
publish_doc('MedScope report - 03 SFT', report)
print(report[:3000])